# 산불발생위치도_전국
https://www.bigdata-forest.kr/orderProduct/FRT000101

In [40]:
import pandas as pd
import numpy as np
from pyproj import Transformer

In [41]:
data = pd.read_csv("temp.csv", encoding="utf-8")

In [42]:
data['OCCRR_DTM'] = pd.to_datetime(data['OCCRR_DTM'].astype(str), format='%Y%m%d%H%M')
data['연도'] = data['OCCRR_DTM'].dt.year
data['월'] = data['OCCRR_DTM'].dt.month
data['일'] = data['OCCRR_DTM'].dt.day
data['시간'] = data['OCCRR_DTM'].dt.hour
data.drop(columns="OCCRR_DTM", inplace=True)

### 2000년도의 화재데이터만 추출

In [43]:
data = data[data["연도"]>=2000]

In [44]:
data['진화소요시간(HH)'] = (data['RQRMN_TM'] // 100) + (data['RQRMN_TM'] % 100 / 60)
data.drop(columns="RQRMN_TM", inplace=True)

In [45]:
data["CTPRV_NM"].unique()

<ArrowStringArray>
[     '서울',   '서울특별시',     '경기도',      '부산',   '부산광역시',    '경상남도',       nan,
      '대구',   '대구광역시',    '경상북도',      '인천',   '인천광역시',      '광주',   '광주광역시',
    '전라남도',      '대전',   '대전광역시',      '울산',   '울산광역시',      '세종',      '충남',
      '충북', '세종특별자치시',    '충청남도',      '경기',     '강원도',    '충청북도',      '강원',
      '전북',      '출북',    '전라북도',      '경남',      '전남',      '경북',   '경북 포항',
      '제주', '제주특별자치도']
Length: 37, dtype: str

In [46]:
data = data[data["CTPRV_NM"]!="전주"]
data = data[data["CTPRV_NM"]!="서부"]

In [47]:
sido_mapping = {
    '서울': '서울특별시',
    '경기': '경기도',
    '부산': '부산광역시',
    '경남': '경상남도',
    '대구': '대구광역시',
    '경북': '경상북도', '경북 포항': '경상북도',
    '인천': '인천광역시',
    '광주': '광주광역시',
    '전남': '전라남도',
    '대전': '대전광역시',
    '울산': '울산광역시',
    '세종': '세종특별자치시',
    '충남': '충청남도',
    '충북': '충청북도', '출북': '충청북도', # 오타 수정
    '강원': '강원특별자치도', '강원도': '강원특별자치도',
    '전북': '전북특별자치도', '전라북도': '전북특별자치도',
    '제주': '제주특별자치도',
}
data['CTPRV_NM'] = data['CTPRV_NM'].replace(sido_mapping)

In [48]:
data.rename(columns={"CTPRV_NM":"발생지역시도명", "SGNG_NM":"발생지역시군구명", "EMNDN_NM":"발생지역읍면동명",
                      "OCCCRR_RI":"발생지역리명", "ARA_LTNMB":"발생지역번지", "CUSE_NM":"발생원인명", 
                      "DMG_AREA":"피해면적(ha)", "DMG_MONEY":"피해금액"}, inplace=True)
data.drop(columns=["OCUR_DYWK", "EXTING_DTM", "ARA_NM"], inplace=True)

In [49]:
transformer = Transformer.from_crs("epsg:5179", "epsg:4326", always_xy=True)

def transform_coordinates(row):
    # X좌표가 경도 방향, Y좌표가 위도 방향에 대응합니다.
    lon, lat = transformer.transform(row['TM_X'], row['TM_Y'])
    return pd.Series([lat, lon], index=['위도', '경도'])
data[['위도', '경도']] = data.apply(transform_coordinates, axis=1)
data.drop(columns=["TM_X", "TM_Y"], inplace=True)

In [50]:
data = data[['연도', '월', '일', '시간', '진화소요시간(HH)', '위도', '경도', 
             '발생지역시도명', '발생지역시군구명', '발생지역읍면동명', '발생지역리명', '발생지역번지', '발생원인명',
             '피해면적(ha)', '피해금액']]

## 결합용 fire_id 만들기 

In [51]:
# 위경도가 없는 쓸모없는 결측치 행 미리 제거
data = data.dropna(subset=['위도', '경도']).copy()
# 2. 강력한 고유 ID(fire_id) 생성 (F_000001, F_000002 ...)
data['fire_id'] = ['F_{:06d}'.format(i) for i in range(len(data))]
# fire_id 컬럼을 맨 앞으로 이동
cols = ['fire_id'] + [c for c in data.columns if c != 'fire_id']
data = data[cols]

In [52]:
data.to_csv("산불발생위치도_전국.csv", encoding="utf-8-sig", index=False)

## 산불_공간DB_위경도

In [53]:
spatial_cols = ['fire_id', '위도', '경도']
df_spatial = data[spatial_cols]
df_spatial.to_csv("산불_공간DB_위경도.csv", index=False, encoding='utf-8-sig')